#### Advanced Computer Vision Techniques

### 0. Requirements

In [85]:
%load_ext autoreload
%autoreload 2

import os

# Set project root
repo_name = "assignment_no_2"
os.chdir(os.getcwd().split(repo_name)[0] + repo_name)
print(f'Changed working directory to: {os.getcwd()}')

# Python modules
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import matplotlib.pyplot as plt

# Local modules
from src.data_loaders import load_cifar10, make_dataloaders
from src.models import build_vgg, build_resnet, build_inception
from src.model_training import train_model
from src.xai import grad_cam
from src.utils import display_images


# ---------- Setup for reproducibility ----------
seed = 28
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Changed working directory to: c:\Users\EMILIO\Desktop\0. Cursos\4. Advanced Computer Vision\assignment_no_2
PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8


### 1. Load Data

In [20]:
train_ds, val_ds, test_ds = load_cifar10()

Training samples: 45000
Validation samples: 5000
Test samples: 10000


### 2. Training pipelines for different models

#### 2.1. Inception Model

##### 2.1.1 Setting Model and Data Loaders

In [ ]:
# --- Experiment Parameters ---
batch_size = 256
model_name = "inception"  # Options: "vgg", "resnet", "inception"
num_classes = 10
epochs = 6

do_finetune = True  # Set to True to enable fine-tuning after feature extraction
lr_feature_extraction = 1e-3
lr_finetune = 5e-5

model = build_inception()

In [ ]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr_feature_extraction)

# Data Loaders
train_loader, val_loader, test_loader, mean, std = make_dataloaders(
    train_ds, val_ds, test_ds, batch_size=batch_size, model_name="inception", augment=True
)

# Get one batch
images, labels = next(iter(train_loader))
display_images(images, labels, mean, std)

✅ mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]


##### 2.1.2 Training (Transfer Learning)

In [ ]:
train_model(model, model_name, train_loader, val_loader, criterion, optimizer, device, epochs)

##### 2.1.3 Training Fine Tunning

In [ ]:
# Command to create enviroment.yaml from conda env
# conda env export > environment.yaml

**9.3 - Fine-Tuning (Optional)**

If `DO_FINETUNE` is `True`, it unfreezes some of the top layers of the base model. It then re-compiles the model with a very low learning rate and trains for a few more epochs. This allows the model to slightly adjust the pre-trained features to better suit the specifics of the CIFAR-10 dataset, often leading to a boost in accuracy. It's crucial that `BatchNormalization` layers remain frozen during this process.

In [ ]:
if DO_FINETUNE:
    print("\n--- Activating Partial Fine-Tuning ---")
    
    # Unfreeze layers based on model architecture, keeping BatchNorm layers frozen
    if MODEL_CHOICE == "vgg":
        fine_tune_at = 15  # Unfreeze from block5_conv1 onwards
        for layer in base_model.layers[fine_tune_at:]:
            layer.trainable = True
    elif MODEL_CHOICE == "resnet":
        for layer in base_model.layers:
            if layer.name.startswith("conv5_block") and not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    else:  # Inception
        for layer in base_model.layers:
            if layer.name.startswith("mixed7") and not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    
    # Re-compile and train with a low learning rate
    print(f"Fine-tuning with learning rate = {LR_FINETUNE}...")
    fine_tune_epochs = max(3, EPOCHS // 2)
    history_ft = compile_and_train(
        model, train_ds, val_ds, 
        epochs=fine_tune_epochs, 
        lr=LR_FINETUNE, 
        model_name=f"{model_name}_finetune"
    )
    
    print("\n--- Evaluating after Fine-Tuning ---")
    evaluate_model(model, test_ds)

**9.4 - Grad-CAM Visualization**

Finally, it takes a sample image from the test set, generates the Grad-CAM overlay using the fully trained model, and saves the resulting image to the `outputs` directory.

In [ ]:
print("\n--- Generating Grad-CAM Visualization ---")
# Take the first image from the original test set and preprocess it
demo_image = (x_test[500] / 255.0).astype(np.float32)

# Resize the image to match the model's expected input size
demo_image_resized = tf.image.resize(demo_image, image_size)
demo_image_resized = tf.cast(demo_image_resized, tf.float32).numpy()

# Generate the overlay using the resized image
overlay = grad_cam(model, demo_image_resized, last_conv_name=last_conv_name, img_size=image_size)

# Save the result
output_filename = f"{model_name}_finetune_gradcam.png" if DO_FINETUNE else f"{model_name}_gradcam.png"
out_path = os.path.join(OUT_DIR, output_filename)
try:
    # The overlay is an RGB float array in [0, 1]. Convert to uint8 for saving
    overlay_uint8 = (overlay * 255).astype(np.uint8)
    imageio.imwrite(out_path, overlay_uint8)
    print(f"Grad-CAM image saved to: {out_path}")
    
    # Display the original image and the Grad-CAM overlay
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    ax1.imshow(demo_image_resized)
    ax1.set_title('Input Image')
    ax1.axis('off')

    ax2.imshow(overlay)
    ax2.set_title('Grad-CAM Overlay')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not save Grad-CAM image. Error: {e}")
